# Customer Churn Prediction

The notebook uses the same leakage-safe training workflow as `train.py`. The raw data is split with stratification before any imputation, scaling, or category encoding is fitted. Random Forest and XGBoost are tuned with five-fold cross-validation, and the selected pipeline is evaluated once on the held-out test set.

## Load the dataset

In [ ]:
import pandas as pd
from train import DATA_PATH, MODEL_PATH, load_dataset, train_model

X, y = load_dataset(DATA_PATH)
print(f"Rows: {len(X):,}; features: {X.shape[1]}")
print("Churn class distribution:")
print(y.value_counts(normalize=True).rename(index={0: "Stay", 1: "Churn"}).round(3))
X.head()

## Train, tune, and evaluate

Preprocessing is part of each model pipeline, so cross-validation fits it independently inside every training fold. Test-set metrics include accuracy, precision, recall, F1-score, ROC-AUC, and the confusion matrix.

In [ ]:
model, best_model_name, cv_results, test_metrics = train_model(DATA_PATH)
print(f"Selected model: {best_model_name}")
display(cv_results)

for metric in ("accuracy", "precision", "recall", "f1", "roc_auc"):
    print(f"{metric.replace('_', ' ').title()}: {test_metrics[metric]:.4f}")

print("Confusion matrix (rows=true, columns=predicted; order: Stay, Churn):")
display(pd.DataFrame(test_metrics["confusion_matrix"], index=["Stay", "Churn"], columns=["Stay", "Churn"]))

## Export the model

The exported joblib file contains the preprocessing and classifier together, ready for the Streamlit app.

In [ ]:
import joblib
import pandas as pd

joblib.dump(model, MODEL_PATH)
print(f"Saved {best_model_name} pipeline to {MODEL_PATH}")